# Lab 2.1 – Structured Output Extractor

## Objective

The objective of this lab is to build an AI-powered Structured Output Extractor using LangChain, Groq, and Pydantic.

The application accepts an unstructured job posting and automatically extracts structured information into a validated Pydantic model.

## Information Extracted

- Job Title
- Company
- Salary Range
- Required Skills
- Location
- Remote Status

## Technologies Used

- Python
- LangChain
- LangChain-Groq
- Pydantic v2
- Groq API
- Jupyter Notebook

# Workflow

The application follows this pipeline:

Unstructured Job Posting

        ↓

Prompt to LLM

        ↓

Groq Llama 3.3

        ↓

Structured Output

        ↓

Pydantic Validation

        ↓

Python Object

In [2]:
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field

from langchain_groq import ChatGroq

In [3]:
load_dotenv()

print("Groq API Loaded:", os.getenv("GROQ_API_KEY") is not None)

Groq API Loaded: True


In [4]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("LLM Initialized Successfully!")

LLM Initialized Successfully!


# Defining the Structured Output Schema

The AI model will generate output that conforms to this schema.

Pydantic validates the generated information automatically.

In [5]:
class JobPosting(BaseModel):

    title: str = Field(description="Job title")

    company: str = Field(description="Company name")

    salary_range: str = Field(description="Salary range")

    required_skills: list[str] = Field(
        description="List of required skills"
    )

    location: str = Field(description="Job location")

    remote_status: str = Field(
        description="Remote, Hybrid or On-site"
    )

# Creating a Structured Output Model

LangChain allows the LLM to directly return a validated Pydantic object instead of plain text.

In [6]:
structured_llm = llm.with_structured_output(JobPosting)

print("Structured Output Pipeline Ready!")

Structured Output Pipeline Ready!


# Sample Unstructured Job Posting

The following text represents an unstructured job advertisement. The AI model will extract the required information and convert it into a structured `JobPosting` object.

In [7]:
job_posting = """
Software Engineer

Company: Google

Location: Islamabad, Pakistan

Salary: PKR 250,000 - PKR 350,000 per month

Required Skills:
- Python
- LangChain
- FastAPI
- Docker
- Git

Work Mode: Hybrid
"""

# Extracting Structured Information

The structured LLM uses the predefined `JobPosting` schema to convert the unstructured text into a validated Python object.

In [8]:
response = structured_llm.invoke(job_posting)

print(response)

title='Software Engineer' company='Google' salary_range='PKR 250,000 - PKR 350,000 per month' required_skills=['Python', 'LangChain', 'FastAPI', 'Docker', 'Git'] location='Islamabad, Pakistan' remote_status='Hybrid'


# Accessing Individual Fields

Since the output is a Pydantic model, each field can be accessed directly as a Python object.

In [9]:
print("Job Title:", response.title)
print("Company:", response.company)
print("Salary:", response.salary_range)
print("Skills:", response.required_skills)
print("Location:", response.location)
print("Remote Status:", response.remote_status)

Job Title: Software Engineer
Company: Google
Salary: PKR 250,000 - PKR 350,000 per month
Skills: ['Python', 'LangChain', 'FastAPI', 'Docker', 'Git']
Location: Islamabad, Pakistan
Remote Status: Hybrid


# Converting the Model to a Dictionary

Pydantic models can easily be converted into dictionaries, making them suitable for databases, APIs, or JSON serialization.

In [10]:
print(response.model_dump())

{'title': 'Software Engineer', 'company': 'Google', 'salary_range': 'PKR 250,000 - PKR 350,000 per month', 'required_skills': ['Python', 'LangChain', 'FastAPI', 'Docker', 'Git'], 'location': 'Islamabad, Pakistan', 'remote_status': 'Hybrid'}


# Converting the Model to JSON

Structured outputs can also be exported as JSON for storage or communication between services.

In [11]:
print(response.model_dump_json(indent=4))

{
    "title": "Software Engineer",
    "company": "Google",
    "salary_range": "PKR 250,000 - PKR 350,000 per month",
    "required_skills": [
        "Python",
        "LangChain",
        "FastAPI",
        "Docker",
        "Git"
    ],
    "location": "Islamabad, Pakistan",
    "remote_status": "Hybrid"
}


# Testing Multiple Job Postings

To evaluate the robustness of the structured output extractor, we will test it on multiple job postings with different formats and requirements.

In [12]:
job_postings = [
    """
    Data Scientist

    Company: Microsoft

    Location: Lahore, Pakistan

    Salary: PKR 300,000 - PKR 450,000

    Required Skills:
    Python
    Machine Learning
    SQL
    TensorFlow

    Work Mode: Remote
    """,

    """
    Backend Developer

    Company: Systems Limited

    Location: Islamabad

    Salary: PKR 180,000 - PKR 250,000

    Required Skills:
    Django
    PostgreSQL
    REST APIs
    Docker

    Work Mode: On-site
    """,

    """
    AI Engineer

    Company: OpenAI

    Location: Remote

    Salary: $120,000 - $170,000

    Required Skills:
    Python
    LangChain
    Pydantic
    FastAPI
    Docker

    Work Mode: Remote
    """
]

In [13]:
for i, posting in enumerate(job_postings, start=1):

    print("=" * 80)
    print(f"Job Posting {i}")

    result = structured_llm.invoke(posting)

    print(result)
    print()

Job Posting 1


title='Data Scientist' company='Microsoft' salary_range='PKR 300,000 - PKR 450,000' required_skills=['Python', 'Machine Learning', 'SQL', 'TensorFlow'] location='Lahore, Pakistan' remote_status='Remote'

Job Posting 2
title='Backend Developer' company='Systems Limited' salary_range='PKR 180,000 - PKR 250,000' required_skills=['Django', 'PostgreSQL', 'REST APIs', 'Docker'] location='Islamabad' remote_status='On-site'

Job Posting 3
title='AI Engineer' company='OpenAI' salary_range='$120,000 - $170,000' required_skills=['Python', 'LangChain', 'Pydantic', 'FastAPI', 'Docker'] location='Remote' remote_status='Remote'



# Displaying Extracted Information

Instead of printing the entire object, we can access each field individually to demonstrate the structured output clearly.

In [14]:
for i, posting in enumerate(job_postings, start=1):

    print("=" * 80)
    print(f"Job Posting {i}")

    result = structured_llm.invoke(posting)

    print(f"Title           : {result.title}")
    print(f"Company         : {result.company}")
    print(f"Salary          : {result.salary_range}")
    print(f"Skills          : {', '.join(result.required_skills)}")
    print(f"Location        : {result.location}")
    print(f"Remote Status   : {result.remote_status}")

    print()

Job Posting 1
Title           : Data Scientist
Company         : Microsoft
Salary          : PKR 300,000 - PKR 450,000
Skills          : Python, Machine Learning, SQL, TensorFlow
Location        : Lahore, Pakistan
Remote Status   : Remote

Job Posting 2
Title           : Backend Developer
Company         : Systems Limited
Salary          : PKR 180,000 - PKR 250,000
Skills          : Django, PostgreSQL, REST APIs, Docker
Location        : Islamabad
Remote Status   : On-site

Job Posting 3
Title           : AI Engineer
Company         : OpenAI
Salary          : $120,000 - $170,000
Skills          : Python, LangChain, Pydantic, FastAPI, Docker
Location        : Remote
Remote Status   : Remote



# Error Handling

Although Pydantic validates structured outputs, it is still good practice to handle unexpected errors when interacting with an LLM or external API.

In [15]:
try:
    result = structured_llm.invoke(job_postings[0])

    print("Extraction Successful!")
    print(result)

except Exception as e:
    print("Error occurred:")
    print(e)

Extraction Successful!
title='Data Scientist' company='Microsoft' salary_range='PKR 300,000 - PKR 450,000' required_skills=['Python', 'Machine Learning', 'SQL', 'TensorFlow'] location='Lahore, Pakistan' remote_status='Remote'


# Observations

During this lab, the following observations were made:

- LangChain successfully converted unstructured job postings into structured data.
- Pydantic validated the extracted information automatically.
- Structured outputs are much easier to process than free-form text.
- The schema ensured consistent field names across different job posting formats.
- This approach can be extended to resumes, invoices, medical reports, and other document types.

# Conclusion

In this lab, I developed a Structured Output Extractor using LangChain, Groq, and Pydantic.

The application accepts unstructured job advertisements and converts them into validated Python objects, making the data easier to process programmatically.

This lab demonstrated how structured outputs improve reliability, consistency, and integration with downstream applications, providing a practical foundation for building production-ready AI systems.

In [16]:
import pandas as pd

rows = []

for posting in job_postings:
    result = structured_llm.invoke(posting)
    rows.append({
        "Title": result.title,
        "Company": result.company,
        "Location": result.location,
        "Salary": result.salary_range,
        "Skills": ", ".join(result.required_skills),
        "Remote": result.remote_status,
    })

df = pd.DataFrame(rows)
df

,Title,Company,Location,Salary,Skills,Remote
0,Data Scientist,Microsoft,"Lahore, Pakistan","PKR 300,000 - PKR 450,000","Python, Machine Learning, SQL, TensorFlow",Remote
1,Backend Developer,Systems Limited,Islamabad,"PKR 180,000 - PKR 250,000","Django, PostgreSQL, REST APIs, Docker",On-site
2,AI Engineer,OpenAI,Remote,"$120,000 - $170,000","Python, LangChain, Pydantic, FastAPI, Docker",Remote
